# Event Study: Market Model CAR using TOPIX500 (equal-weight)

このノートブックでは、`risk_change_scores`（または `risk_novelty_scores`）で作ったスコアと、
有報提出日（`submit_date`）前後の株価反応を **市場モデル（Market Model）** で推定したCARで検証します。

## 市場調整（Market Model）
- 推定期間（例：-120〜-21営業日）で
  \(R_{i,t} = \alpha_i + \beta_i R_{m,t} + \epsilon_{i,t}\) を推定
- イベント期間（例：[-1,+1]）で
  \(AR_{i,t} = R_{i,t} - (\hat\alpha_i + \hat\beta_i R_{m,t})\)
- \(CAR = \sum AR\)

## 市場リターン \(R_m\)
- 年別 `topix_companies_YYYY.csv` から **TOPIX Core30 + Large70 + Mid400** を採用（≒TOPIX500）
- それらの銘柄の日次リターンを等加重平均して \(R_m\) を作る


In [1]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm

tqdm.pandas()


In [3]:
def find_project_root(start_dir: str) -> str:
    d = os.path.abspath(start_dir)
    while True:
        if os.path.exists(os.path.join(d, "docker-compose.yml")) and os.path.exists(os.path.join(d, "requirements.txt")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start_dir)
        d = parent

PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, '.env'))

DB_USER = os.getenv('POSTGRES_USER')
DB_PASS = os.getenv('POSTGRES_PASSWORD')
DB_HOST = os.getenv('POSTGRES_HOST', 'localhost')
DB_PORT = os.getenv('POSTGRES_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB')

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print('Connected')


PROJECT_ROOT: /home/junxzi/dev/risk-novelty-stock-jp
Connected


In [4]:
# --- Config ---
SAMPLE_MODE = True
SAMPLE_N = 200
SAMPLE_SEED = 42

EVENT_WINDOWS = {
    'm1_p1': (-1, 1),
    '0_p1': (0, 1),
    'm3_p3': (-3, 3),
}

EST_WINDOW = (-120, -21)  # trading days relative to event day
CUTOFF_HOUR = 15

TOPIX500_CATEGORIES = {'TOPIX Core30', 'TOPIX Large70', 'TOPIX Mid400'}


In [5]:
# 1) Load change score sample + metadata
q = """
SELECT r.company_id::uuid AS company_id,
       r.fiscal_year,
       r.doc_id_curr,
       r.change_topk
FROM risk_change_scores r;
"""

df = pd.read_sql(q, engine)

if SAMPLE_MODE:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=SAMPLE_SEED)

q_meta = """
SELECT d.doc_id, d.submit_date, c.security_code
FROM edinet_documents d
JOIN companies c ON c.company_id = d.company_id
WHERE d.doc_id = ANY(:doc_ids);
"""

doc_ids = df['doc_id_curr'].dropna().unique().tolist()
meta = pd.read_sql(text(q_meta), engine, params={'doc_ids': doc_ids})
meta['submit_ts'] = pd.to_datetime(meta['submit_date'])
meta['submit_date_only'] = meta['submit_ts'].dt.floor('D')
meta['submit_year'] = meta['submit_ts'].dt.year
meta['security_code'] = meta['security_code'].astype(str)

df = df.merge(meta[['doc_id','submit_ts','submit_date_only','submit_year','security_code']], left_on='doc_id_curr', right_on='doc_id', how='left')
print('rows with security_code:', df['security_code'].notna().mean())
df.head()


rows with security_code: 1.0


,company_id,fiscal_year,doc_id_curr,change_topk,doc_id,submit_ts,submit_date_only,submit_year,security_code
0,be377664-129e-4752-9f94-a26b7d4efe28,2023,S100R1IZ,0.059111,S100R1IZ,2023-06-29 09:34:00,2023-06-29,2023,95090
1,adecbfd9-481f-4a13-a662-3fc162a55c89,2024,S100TQ3L,0.045933,S100TQ3L,2024-06-25 14:19:00,2024-06-25,2024,28100
2,eba75e57-40e8-4fab-a89a-ebff3e4bab5a,2025,S100VIKQ,0.091050,S100VIKQ,2025-03-28 15:32:00,2025-03-28,2025,49670
3,7a284887-8d57-4023-9ea5-9c5359ab14db,2021,S100LJIS,0.031498,S100LJIS,2021-06-21 14:13:00,2021-06-21,2021,79660
4,602f7941-6045-4f05-91d5-ec4b57cc3c52,2022,S100ODMQ,0.119628,S100ODMQ,2022-06-23 13:25:00,2022-06-23,2022,28020


In [10]:
# 2) Build TOPIX500 membership by year (≈ Core30 + Large70 + Mid400)

def get_topix500_codes(year: int) -> list[str]:
    path = os.path.join('data', 'processed', f'topix_companies_{year}.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    t = pd.read_csv(path, dtype={'security_code': str, 'scale_category': str})
    t = t[t['scale_category'].isin(TOPIX500_CATEGORIES)].copy()
    # security_code is like 13320 (5 chars)
    return sorted(t['security_code'].astype(str).unique().tolist())

needed_years = sorted(df['submit_year'].dropna().astype(int).unique().tolist())
print('needed_years:', needed_years)

codes_by_year = {y: get_topix500_codes(y) for y in needed_years}
print({y: len(v) for y,v in codes_by_year.items()})


needed_years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
{2018: 501, 2019: 500, 2020: 499, 2021: 500, 2022: 500, 2023: 498, 2024: 501, 2025: 497}


In [11]:
# 3) Fetch quotes for (a) sample stocks, (b) market portfolio constituents

# date range needed (rough): estimation window ~ 150 trading days before event
min_day = df['submit_date_only'].min() - pd.Timedelta(days=260)
max_day = df['submit_date_only'].max() + pd.Timedelta(days=10)

stock_codes = sorted(df['security_code'].dropna().astype(str).unique().tolist())
market_codes = sorted(set(sum((codes_by_year[y] for y in needed_years), [])))

print('sample stock codes:', len(stock_codes))
print('market codes union:', len(market_codes))

def fetch_quotes(codes: list[str]) -> pd.DataFrame:
    q = """
    SELECT code, date, adjusted_close
    FROM daily_quotes
    WHERE code = ANY(:codes)
      AND date BETWEEN :d0 AND :d1
    ORDER BY code, date;
    """
    out = pd.read_sql(text(q), engine, params={'codes': codes, 'd0': min_day.date(), 'd1': max_day.date()})
    out['date'] = pd.to_datetime(out['date'])
    out['adjusted_close'] = pd.to_numeric(out['adjusted_close'], errors='coerce')
    return out.dropna(subset=['adjusted_close'])

quotes_stock = fetch_quotes(stock_codes)
quotes_mkt = fetch_quotes(market_codes)

print('quotes_stock rows:', len(quotes_stock))
print('quotes_mkt rows:', len(quotes_mkt))


sample stock codes: 184
market codes union: 583
quotes_stock rows: 326138
quotes_mkt rows: 1089353


In [12]:
# 4) Compute daily returns for stocks and market portfolio

def add_returns(q: pd.DataFrame) -> pd.DataFrame:
    q = q.sort_values(['code','date']).copy()
    q['ret'] = q.groupby('code')['adjusted_close'].pct_change()
    return q.dropna(subset=['ret'])

rets_stock = add_returns(quotes_stock)
rets_mkt_all = add_returns(quotes_mkt)

# Market return: equal-weight mean among TOPIX500-ish constituents of that YEAR
rets_mkt_all['year'] = rets_mkt_all['date'].dt.year

# tag membership
membership = []
for y, codes in codes_by_year.items():
    membership.append(pd.DataFrame({'year': y, 'code': codes}))
mem = pd.concat(membership, ignore_index=True)

rets_mkt = rets_mkt_all.merge(mem, on=['year','code'], how='inner')
rm = rets_mkt.groupby('date')['ret'].mean().rename('rm').reset_index()

print('Rm days:', len(rm))
rm.head()


Rm days: 1829


,date,rm
0,2018-01-04,0.023626
1,2018-01-05,0.008108
2,2018-01-09,0.005259
3,2018-01-10,0.000401
4,2018-01-11,-0.001559


In [13]:
# 5) Helpers: trading day alignment + market model estimation

# create per-code trading calendar
stock_by_code = {c: g.sort_values('date').reset_index(drop=True) for c,g in rets_stock.groupby('code')}
rm_series = rm.set_index('date')['rm']

def next_trading_day_stock(code: str, day: pd.Timestamp) -> pd.Timestamp | None:
    g = stock_by_code.get(code)
    if g is None or g.empty:
        return None
    idx = g['date'].searchsorted(day, side='left')
    if idx >= len(g):
        return None
    return g.loc[idx, 'date']

def shift_trading_day_stock(code: str, day: pd.Timestamp, k: int) -> pd.Timestamp | None:
    g = stock_by_code.get(code)
    if g is None or g.empty:
        return None
    idx = g['date'].searchsorted(day, side='left')
    if idx >= len(g):
        return None
    idx2 = idx + k
    if idx2 < 0 or idx2 >= len(g):
        return None
    return g.loc[idx2, 'date']

def estimate_alpha_beta(ri: pd.Series, rm: pd.Series) -> tuple[float,float] | None:
    # align
    df_ab = pd.concat([ri.rename('ri'), rm.rename('rm')], axis=1).dropna()
    if len(df_ab) < 30:
        return None
    x = df_ab['rm'].values
    y = df_ab['ri'].values
    var = np.var(x)
    if var <= 1e-12:
        return None
    beta = np.cov(x, y, ddof=0)[0,1] / var
    alpha = float(np.mean(y) - beta * np.mean(x))
    return alpha, float(beta)

def car_market_model(code: str, submit_ts: pd.Timestamp, window: tuple[int,int]) -> dict:
    if pd.isna(submit_ts):
        return {'car': np.nan, 'alpha': np.nan, 'beta': np.nan, 'n_est': 0}

    base = pd.Timestamp(submit_ts.date())
    event_day = next_trading_day_stock(code, base)
    if event_day is None:
        return {'car': np.nan, 'alpha': np.nan, 'beta': np.nan, 'n_est': 0}

    if submit_ts.hour >= CUTOFF_HOUR:
        event_day = shift_trading_day_stock(code, event_day, 1)
        if event_day is None:
            return {'car': np.nan, 'alpha': np.nan, 'beta': np.nan, 'n_est': 0}

    # build estimation and event date lists
    est_start = shift_trading_day_stock(code, event_day, EST_WINDOW[0])
    est_end = shift_trading_day_stock(code, event_day, EST_WINDOW[1])
    ev_start = shift_trading_day_stock(code, event_day, window[0])
    ev_end = shift_trading_day_stock(code, event_day, window[1])
    if None in (est_start, est_end, ev_start, ev_end):
        return {'car': np.nan, 'alpha': np.nan, 'beta': np.nan, 'n_est': 0}

    g = stock_by_code.get(code)
    g = g.set_index('date')
    ri = g['ret']

    est_idx = ri.loc[est_start:est_end].index
    ev_idx = ri.loc[ev_start:ev_end].index

    alpha_beta = estimate_alpha_beta(ri.loc[est_idx], rm_series.reindex(est_idx))
    if alpha_beta is None:
        return {'car': np.nan, 'alpha': np.nan, 'beta': np.nan, 'n_est': 0}

    alpha, beta = alpha_beta
    # abnormal returns in event window
    rm_ev = rm_series.reindex(ev_idx)
    ri_ev = ri.reindex(ev_idx)
    ar = ri_ev - (alpha + beta * rm_ev)
    car = float(ar.dropna().sum())

    return {'car': car, 'alpha': alpha, 'beta': beta, 'n_est': int(len(est_idx))}


In [14]:
# 6) Compute CARs for each window

rows = []
for r in tqdm(df.itertuples(index=False), total=len(df), desc='Event study'):
    if pd.isna(r.security_code) or pd.isna(r.submit_ts):
        continue
    code = str(r.security_code)
    for wname, w in EVENT_WINDOWS.items():
        res = car_market_model(code, pd.to_datetime(r.submit_ts), w)
        rows.append({
            'company_id': r.company_id,
            'fiscal_year': int(r.fiscal_year),
            'doc_id_curr': r.doc_id_curr,
            'security_code': code,
            'submit_ts': pd.to_datetime(r.submit_ts),
            'window': wname,
            'car_mm': res['car'],
            'alpha': res['alpha'],
            'beta': res['beta'],
            'n_est': res['n_est'],
            'change_topk': float(r.change_topk) if not pd.isna(r.change_topk) else np.nan,
        })

car_df = pd.DataFrame(rows)
print('rows:', len(car_df))
car_df.head()


Event study:   0%|          | 0/200 [00:00<?, ?it/s]

rows: 600


,company_id,fiscal_year,doc_id_curr,security_code,submit_ts,window,car_mm,alpha,beta,n_est,change_topk
0,be377664-129e-4752-9f94-a26b7d4efe28,2023,S100R1IZ,95090,2023-06-29 09:34:00,m1_p1,-0.015338,0.001476,0.607574,100,0.059111
1,be377664-129e-4752-9f94-a26b7d4efe28,2023,S100R1IZ,95090,2023-06-29 09:34:00,0_p1,-0.016594,0.001476,0.607574,100,0.059111
2,be377664-129e-4752-9f94-a26b7d4efe28,2023,S100R1IZ,95090,2023-06-29 09:34:00,m3_p3,-0.054339,0.001476,0.607574,100,0.059111
3,adecbfd9-481f-4a13-a662-3fc162a55c89,2024,S100TQ3L,28100,2024-06-25 14:19:00,m1_p1,-0.002975,-0.001101,0.545587,100,0.045933
4,adecbfd9-481f-4a13-a662-3fc162a55c89,2024,S100TQ3L,28100,2024-06-25 14:19:00,0_p1,-0.002773,-0.001101,0.545587,100,0.045933


In [15]:
# 7) Quick check: correlation by window (market model CAR)
for wname in EVENT_WINDOWS.keys():
    sub = car_df[(car_df['window']==wname)].dropna(subset=['car_mm','change_topk']).copy()
    corr = sub[['change_topk','car_mm']].corr().iloc[0,1]
    print(wname, 'n=', len(sub), 'corr=', corr)

car_df.groupby('window')['car_mm'].describe()


m1_p1 n= 197 corr= -0.03297388533196171
0_p1 n= 197 corr= 0.005175266941250951
m3_p3 n= 194 corr= -0.02110758318759193


,count,mean,std,min,25%,50%,75%,max
window,,,,,,,,
0_p1,197.0,-0.001993,0.037724,-0.175880,-0.016811,-0.002099,0.011374,0.199535
m1_p1,197.0,-0.000383,0.044529,-0.168098,-0.020011,0.000436,0.012255,0.200195
m3_p3,194.0,-0.006292,0.060458,-0.345850,-0.033542,-0.002688,0.021166,0.163241
